In [1]:
# Install required libraries
!pip install openai pandas sentence-transformers -q

In [2]:
import pandas as pd
import json
import time
from datetime import datetime

# ============================================
# MULTI-AGENT REAL ESTATE AI SYSTEM
# ============================================
# Agent 1: Data Collector Agent
# Agent 2: Analysis Agent
# Agent 3: Matching Agent
# Agent 4: Orchestrator Agent (coordinates all)
# ============================================

print("🤖 Real Estate Multi-Agent System Initializing...")
print("=" * 50)

# Shared memory between agents
shared_memory = {
    "properties": [],
    "analysis": {},
    "matches": [],
    "logs": []
}

def log_agent(agent_name, message):
    timestamp = datetime.now().strftime("%H:%M:%S")
    log = f"[{timestamp}] [{agent_name}] {message}"
    shared_memory["logs"].append(log)
    print(log)

print("✅ System initialized successfully")
print("✅ Shared memory created")
print("✅ 4 Agents ready to deploy")

🤖 Real Estate Multi-Agent System Initializing...
✅ System initialized successfully
✅ Shared memory created
✅ 4 Agents ready to deploy


In [3]:
# ============================================
# AGENT 1: DATA COLLECTOR AGENT
# ============================================

def data_collector_agent():
    log_agent("DATA COLLECTOR AGENT", "Starting property data collection...")

    properties = [
        {"id": "P001", "title": "Modern Apartment in New Cairo", "price_egp": 2500000, "location": "New Cairo, Cairo", "bedrooms": 3, "bathrooms": 2, "area_sqm": 150, "type": "Apartment"},
        {"id": "P002", "title": "Villa in Sheikh Zayed", "price_egp": 8500000, "location": "Sheikh Zayed, Giza", "bedrooms": 5, "bathrooms": 4, "area_sqm": 400, "type": "Villa"},
        {"id": "P003", "title": "Studio in Maadi", "price_egp": 950000, "location": "Maadi, Cairo", "bedrooms": 1, "bathrooms": 1, "area_sqm": 60, "type": "Studio"},
        {"id": "P004", "title": "Penthouse in Zamalek", "price_egp": 12000000, "location": "Zamalek, Cairo", "bedrooms": 4, "bathrooms": 3, "area_sqm": 280, "type": "Penthouse"},
        {"id": "P005", "title": "Apartment in 6th of October", "price_egp": 1800000, "location": "6th of October, Giza", "bedrooms": 2, "bathrooms": 1, "area_sqm": 110, "type": "Apartment"},
        {"id": "P006", "title": "Twin House in Madinaty", "price_egp": 5200000, "location": "Madinaty, Cairo", "bedrooms": 4, "bathrooms": 3, "area_sqm": 320, "type": "Twin House"},
        {"id": "P007", "title": "Chalet in North Coast", "price_egp": 3100000, "location": "North Coast, Alexandria", "bedrooms": 3, "bathrooms": 2, "area_sqm": 180, "type": "Chalet"},
        {"id": "P008", "title": "Villa in New Capital", "price_egp": 9800000, "location": "New Capital, Cairo", "bedrooms": 6, "bathrooms": 5, "area_sqm": 500, "type": "Villa"},
    ]

    shared_memory["properties"] = properties
    log_agent("DATA COLLECTOR AGENT", f"✅ Collected {len(properties)} properties successfully")
    log_agent("DATA COLLECTOR AGENT", "📤 Sending data to Analysis Agent...")
    return properties

properties = data_collector_agent()

[15:52:36] [DATA COLLECTOR AGENT] Starting property data collection...
[15:52:36] [DATA COLLECTOR AGENT] ✅ Collected 8 properties successfully
[15:52:36] [DATA COLLECTOR AGENT] 📤 Sending data to Analysis Agent...


In [4]:
# ============================================
# AGENT 2: ANALYSIS AGENT
# ============================================

def analysis_agent(properties):
    log_agent("ANALYSIS AGENT", "Starting market analysis...")

    df = pd.DataFrame(properties)

    # Price per sqm
    df['price_per_sqm'] = (df['price_egp'] / df['area_sqm']).round(2)

    # Market tier
    def get_tier(price):
        if price < 2000000:
            return 'Affordable'
        elif price < 5000000:
            return 'Mid-Range'
        elif price < 10000000:
            return 'Premium'
        else:
            return 'Luxury'

    df['market_tier'] = df['price_egp'].apply(get_tier)

    analysis = {
        "total_properties": len(df),
        "avg_price": df['price_egp'].mean(),
        "avg_price_per_sqm": df['price_per_sqm'].mean(),
        "most_common_type": df['type'].mode()[0],
        "tier_distribution": df['market_tier'].value_counts().to_dict(),
        "city_distribution": df['location'].apply(lambda x: x.split(',')[-1].strip()).value_counts().to_dict()
    }

    shared_memory["analysis"] = analysis
    shared_memory["properties"] = df.to_dict('records')

    log_agent("ANALYSIS AGENT", f"✅ Analysis complete")
    log_agent("ANALYSIS AGENT", f"📊 Avg Price: {analysis['avg_price']:,.0f} EGP")
    log_agent("ANALYSIS AGENT", f"📊 Avg Price/sqm: {analysis['avg_price_per_sqm']:,.0f} EGP")
    log_agent("ANALYSIS AGENT", f"📊 Tier Distribution: {analysis['tier_distribution']}")
    log_agent("ANALYSIS AGENT", "📤 Sending results to Matching Agent...")

    return analysis

analysis = analysis_agent(properties)

[15:54:27] [ANALYSIS AGENT] Starting market analysis...
[15:54:27] [ANALYSIS AGENT] ✅ Analysis complete
[15:54:27] [ANALYSIS AGENT] 📊 Avg Price: 5,481,250 EGP
[15:54:27] [ANALYSIS AGENT] 📊 Avg Price/sqm: 20,755 EGP
[15:54:27] [ANALYSIS AGENT] 📊 Tier Distribution: {'Premium': 3, 'Mid-Range': 2, 'Affordable': 2, 'Luxury': 1}
[15:54:27] [ANALYSIS AGENT] 📤 Sending results to Matching Agent...


In [5]:
# ============================================
# AGENT 3: MATCHING AGENT
# ============================================

def matching_agent(properties, buyer_profile):
    log_agent("MATCHING AGENT", f"Finding matches for buyer: {buyer_profile['name']}...")

    matches = []

    for prop in properties:
        score = 0
        reasons = []

        # Budget check
        if prop['price_egp'] <= buyer_profile['max_budget']:
            score += 40
            reasons.append("✅ Within budget")
        else:
            reasons.append("❌ Over budget")
            continue

        # Bedrooms check
        if prop['bedrooms'] >= buyer_profile['min_bedrooms']:
            score += 30
            reasons.append("✅ Enough bedrooms")

        # Location check
        if buyer_profile['preferred_city'].lower() in prop['location'].lower():
            score += 30
            reasons.append("✅ Preferred location")

        matches.append({
            "property_id": prop['id'],
            "title": prop['title'],
            "price_egp": prop['price_egp'],
            "match_score": score,
            "reasons": reasons
        })

    # Sort by score
    matches = sorted(matches, key=lambda x: x['match_score'], reverse=True)
    shared_memory["matches"] = matches

    log_agent("MATCHING AGENT", f"✅ Found {len(matches)} matching properties")
    for m in matches[:3]:
        log_agent("MATCHING AGENT", f"🏠 {m['title']} | Score: {m['match_score']}% | {', '.join(m['reasons'])}")

    log_agent("MATCHING AGENT", "📤 Sending matches to Orchestrator...")
    return matches

# Sample buyer profile
buyer_profile = {
    "name": "Ahmed Hassan",
    "max_budget": 6000000,
    "min_bedrooms": 3,
    "preferred_city": "Cairo"
}

matches = matching_agent(shared_memory["properties"], buyer_profile)

[15:55:14] [MATCHING AGENT] Finding matches for buyer: Ahmed Hassan...
[15:55:14] [MATCHING AGENT] ✅ Found 5 matching properties
[15:55:14] [MATCHING AGENT] 🏠 Modern Apartment in New Cairo | Score: 100% | ✅ Within budget, ✅ Enough bedrooms, ✅ Preferred location
[15:55:14] [MATCHING AGENT] 🏠 Twin House in Madinaty | Score: 100% | ✅ Within budget, ✅ Enough bedrooms, ✅ Preferred location
[15:55:14] [MATCHING AGENT] 🏠 Studio in Maadi | Score: 70% | ✅ Within budget, ✅ Preferred location
[15:55:14] [MATCHING AGENT] 📤 Sending matches to Orchestrator...


In [6]:
# ============================================
# AGENT 4: ORCHESTRATOR AGENT
# ============================================

def orchestrator_agent():
    log_agent("ORCHESTRATOR", "🚀 Starting full pipeline...")
    print("=" * 50)

    # Step 1: Collect data
    log_agent("ORCHESTRATOR", "📋 Step 1: Calling Data Collector Agent...")
    properties = data_collector_agent()
    print("=" * 50)

    # Step 2: Analyze data
    log_agent("ORCHESTRATOR", "📋 Step 2: Calling Analysis Agent...")
    analysis = analysis_agent(properties)
    print("=" * 50)

    # Step 3: Match properties
    log_agent("ORCHESTRATOR", "📋 Step 3: Calling Matching Agent...")
    matches = matching_agent(shared_memory["properties"], buyer_profile)
    print("=" * 50)

    # Final report
    log_agent("ORCHESTRATOR", "📋 Step 4: Generating Final Report...")
    print("\n")
    print("=" * 50)
    print("📊 FINAL ORCHESTRATOR REPORT")
    print("=" * 50)
    print(f"✅ Total Properties Collected: {analysis['total_properties']}")
    print(f"✅ Average Market Price: {analysis['avg_price']:,.0f} EGP")
    print(f"✅ Most Common Type: {analysis['most_common_type']}")
    print(f"✅ Total Matches Found: {len(matches)}")
    print(f"\n🏆 Top Match for {buyer_profile['name']}:")
    if matches:
        top = matches[0]
        print(f"   Property: {top['title']}")
        print(f"   Price: {top['price_egp']:,.0f} EGP")
        print(f"   Match Score: {top['match_score']}%")
    print("=" * 50)
    print("\n📝 FULL AGENT LOGS:")
    for log in shared_memory["logs"]:
        print(log)

orchestrator_agent()

[15:56:01] [ORCHESTRATOR] 🚀 Starting full pipeline...
[15:56:01] [ORCHESTRATOR] 📋 Step 1: Calling Data Collector Agent...
[15:56:01] [DATA COLLECTOR AGENT] Starting property data collection...
[15:56:01] [DATA COLLECTOR AGENT] ✅ Collected 8 properties successfully
[15:56:01] [DATA COLLECTOR AGENT] 📤 Sending data to Analysis Agent...
[15:56:01] [ORCHESTRATOR] 📋 Step 2: Calling Analysis Agent...
[15:56:01] [ANALYSIS AGENT] Starting market analysis...
[15:56:01] [ANALYSIS AGENT] ✅ Analysis complete
[15:56:01] [ANALYSIS AGENT] 📊 Avg Price: 5,481,250 EGP
[15:56:01] [ANALYSIS AGENT] 📊 Avg Price/sqm: 20,755 EGP
[15:56:01] [ANALYSIS AGENT] 📊 Tier Distribution: {'Premium': 3, 'Mid-Range': 2, 'Affordable': 2, 'Luxury': 1}
[15:56:01] [ANALYSIS AGENT] 📤 Sending results to Matching Agent...
[15:56:01] [ORCHESTRATOR] 📋 Step 3: Calling Matching Agent...
[15:56:01] [MATCHING AGENT] Finding matches for buyer: Ahmed Hassan...
[15:56:01] [MATCHING AGENT] ✅ Found 5 matching properties
[15:56:01] [MATCHING